# Lib

In [ ]:
import torch

from mpc import mpc
from mpc.mpc import QuadCost, LinDx, GradMethods
from mpc.env_dx import cartpole

import numpy as np
import numpy.random as npr

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F

import os
import io
import base64
import tempfile
from IPython.display import HTML

from tqdm import tqdm

%matplotlib inline

print("GOGOGO!")

# Utils

## 初始化

In [96]:
# 均匀分布采样
def uniform(shape, low, high):
    r = high - low
    return torch.rand(shape) * r + low

# 生成初始状态，来自mpc torch
def cartpole_initx(n_batch, angle=180):
    ratio = angle / 180.0
    th = uniform(n_batch, -ratio*np.pi, ratio*np.pi)
    thdot = uniform(n_batch, -.5 * ratio, .5 * ratio)
    x = uniform(n_batch, -0.5 * ratio, 0.5 * ratio)
    xdot = uniform(n_batch, -0.5 * ratio, 0.5 * ratio)
    xinit = torch.stack((x, xdot, torch.cos(th), torch.sin(th), thdot), dim=1)
    ref = torch.tensor([0., 0., 1., 0., 0.])
    return xinit - ref

# 获得用于mpc的cost函数
def get_cost(q, r, f, mpc_T, n_batch, dtype, device):
    """
    Inputs:
        q: nn.Parameter, shape: (n_state,)
        r: tensor, shape: (n_control,)
        f: nn.Parameter, shape: (n_state, n_state)
    Outputs:
        Q: tensor, no grad, shape: (mpc_T, n_batch, n_state, n_state)
    """
    # Create a copy with requires_grad=False
    q_ = q.detach().clone()
    q_.requires_grad = False
    f_ = f.detach().clone()
    f_.requires_grad = False

    # Combine control and state costs
    q_comb = torch.cat((q_, r), dim=0)  # shape: (n_state + n_control,)
    # Zero padding for terminal costs to (n_state + n_control, n_state + n_control)
    f_comb = torch.cat((f_, torch.ones(1, 5) * 2), dim=0)
    f_comb = torch.cat((f_comb, torch.ones(6, 1) * 2), dim=1)

    # Create Q matrices
    Q = torch.diag(q_comb).unsqueeze(0).unsqueeze(0).repeat(mpc_T, n_batch, 1, 1)
    # Modify last time step Q matrix to include terminal cost
    Q[-1, :, :, :] = f_comb.unsqueeze(0).unsqueeze(0).repeat(1, n_batch, 1, 1)
    return Q

# q = nn.Parameter(torch.tensor([1., 1.]))
# r = torch.tensor([1.])
# f = nn.Parameter(torch.randn(5, 5)))
# mpc_T = 10
# get_cost(q, r, f, mpc_T, 1)

In [97]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

## MPC

In [98]:
class myCartpole(cartpole.CartpoleDx):
    def __init__(self, params=None):
        super().__init__()

    def forward(self, state, u):
        squeeze = state.ndimension() == 1

        if squeeze:
            state = state.unsqueeze(0)
            u = u.unsqueeze(0)

        if state.is_cuda and not self.params.is_cuda:
            self.params = self.params.cuda()
        state = state + self.goal_state # Change
        gravity, masscart, masspole, length = torch.unbind(self.params)
        total_mass = masspole + masscart
        polemass_length = masspole * length

        u = torch.clamp(u[:,0], -self.force_mag, self.force_mag)

        x, dx, cos_th, sin_th, dth = torch.unbind(state, dim=1)
        th = torch.atan2(sin_th, cos_th)

        cart_in = (u + polemass_length * dth**2 * sin_th) / total_mass
        th_acc = (gravity * sin_th - cos_th * cart_in) / \
                 (length * (4./3. - masspole * cos_th**2 /
                                     total_mass))
        xacc = cart_in - polemass_length * th_acc * cos_th / total_mass

        x = x + self.dt * dx
        dx = dx + self.dt * xacc
        th = th + self.dt * dth
        dth = dth + self.dt * th_acc

        state = torch.stack((
            x, dx, torch.cos(th), torch.sin(th), dth
        ), 1)

        return state - self.goal_state # Change

    def get_frame(self, state, ax=None):
        state = util.get_data_maybe(state.view(-1))
        assert len(state) == 5
        state = state + self.goal_state # Change
        x, dx, cos_th, sin_th, dth = torch.unbind(state)
        gravity, masscart, masspole, length = torch.unbind(self.params)
        th = np.arctan2(sin_th, cos_th)
        th_x = sin_th*length
        th_y = cos_th*length

        if ax is None:
            fig, ax = plt.subplots(figsize=(6,6))
        else:
            fig = ax.get_figure()
        ax.plot((x,x+th_x), (0, th_y), color='k')
        ax.set_xlim((-length*2, length*2))
        ax.set_ylim((-length*2, length*2))
        return fig, ax

# testEnv = myCartpole()
# testX = cartpole_initx(4)
# testU = torch.zeros((4, 1))
# print(testX)
# print(testEnv.forward(testX, testU))

In [99]:
def solve_mpc_cartpole(n_batch, T, mpc_T, q, r, f, DTYPE, DEVICE):
    dx = myCartpole()
    t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

    Q = get_cost(q, r, f, mpc_T, n_batch, dtype=DTYPE, device=DEVICE)
    p = torch.zeros(dx.n_state + 1, dtype=DTYPE, device=DEVICE).unsqueeze(0).repeat(mpc_T, n_batch, 1)

    x_list = []
    u_list = []
    objs = []

    x = cartpole_initx(n_batch)
    u_init = None
    for t in tqdm(range(T)):
        nominal_states, nominal_actions, nominal_objs = mpc.MPC(
            dx.n_state, dx.n_ctrl, mpc_T,
            u_init=u_init,                                  # u_init的作用: warm-start，表示对控制序列的初始猜测 (用上一个时刻的预测结果)，可以加速收敛
            # u_lower=dx.lower, u_upper=dx.upper,
            lqr_iter=50,
            verbose=0,
            exit_unconverged=False,
            detach_unconverged=False,
            linesearch_decay=dx.linesearch_decay,
            max_linesearch_iter=dx.max_linesearch_iter,
            grad_method=GradMethods.AUTO_DIFF,
            eps=1e-2,
        )(x, QuadCost(Q, p), dx)
        
        x_list.append(nominal_states)
        u_list.append(nominal_actions)
        objs.append(nominal_objs)

        next_action = nominal_actions[0]
        u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
        u_init[-2] = u_init[-3] # 不知道为什么这样写，猜想应该是u[-1] = u[-2]才对

        x = dx(x, next_action) # 往前走一步

        # 以下都是用来可视化的
        # n_col = 4
        # n_row = n_batch // n_col
        # fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
        # axs = axs.reshape(-1)
        # for i in range(n_batch):
        #     dx.get_frame(x[i], ax=axs[i])
        #     axs[i].get_xaxis().set_visible(False)
        #     axs[i].get_yaxis().set_visible(False)
        # fig.tight_layout()
        # fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
        # plt.close(fig)
        
    # action_history = torch.stack(action_history).detach()[:,:,0]
    return x_list, u_list, objs

## RDP

## 激活函数

In [100]:
def max_square(x, eps):
    return torch.max(0, (x - eps).pow(2))

### 算V

In [101]:
def cost_cartpole(x, u, Q, R, F, is_terminal):
    if not is_terminal:
        return 0.5 * torch.matmul(torch.matmul(x.unsqueeze(1), Q), x.unsqueeze(2)).squeeze(1).squeeze(1) + \
               0.5 * torch.matmul(torch.matmul(u.unsqueeze(1), R), u.unsqueeze(2)).squeeze(1).squeeze(1) + \
               0.5 * torch.matmul(torch.matmul(x.unsqueeze(1), F), x.unsqueeze(2)).squeeze(1).squeeze(1)
    else:
        return 0.5 * torch.matmul(torch.matmul(x.unsqueeze(1), Q), x.unsqueeze(2)).squeeze(1).squeeze(1) + \
               0.5 * torch.matmul(torch.matmul(x.unsqueeze(1), F), x.unsqueeze(2)).squeeze(1).squeeze(1)

def cal_V(states, actions, q, r, f, mpc_T, T):
    Q = torch.diag(q)
    R = r
    F = f
    V_list = []
    for t in range(mpc_T):
        V_now = 0
        for i in range(T):
            if i == T - 1:
                V_now += cost_cartpole(states[t][i], actions[t][i], Q, R, F, True)
            else:
                V_now += cost_cartpole(states[t][i], actions[t][i], Q, R, F, False)
        V_list.append(V_now)
    return V_list


### RDP loss

In [102]:
def RDP_loss(VN_list, x_list, u_list, q, r, f, alpha, mpc_T, func, test=False, log_path=None):
    """
    Input:
        VN_list: list of the cost function value
        x_list: list of the state
        u_list: list of the control
        alpha: the weight of the running cost term
        func: the function used to incoporate the RDP inequality into the loss function
        test: whether to print the RDP value
    Return:
        RDP criteria
    """
    Q = torch.diag(q)
    R = r
    F = f
    loss = 0
    for i in range(mpc_T - 1):
        RDP = VN_list[i+1] + alpha * cost_cartpole(x_list[i], u_list[i], Q, R, F, False) - VN_list[i] # Wish RDP <= 0

        if test:
            if log_path is not None:
                with open(log_path, 'a') as f:
                    f.write(f'RDP{i}: {RDP}\n')
        loss += func(RDP) 
        
    return loss

### 权重正则loss

In [103]:
def weight_loss(q, f, p_lambda=1.):
    penalty = (q.sum() - 1.).pow(2) + (f.sum() - 1.).pow(2)
    return p_lambda * penalty

# q = nn.Parameter(torch.tensor([1., 1.]))
# f = nn.Parameter(torch.ones(2, 2))
# print(weight_loss(q, f))

### Lyap loss

In [104]:
def target_contraint():
    pass

def Lyap_loss(VN_list, x_list, u_list, q, r, f, alpha, mpc_T, func, test=False, log_path=None):
    """
    Input:
        VN_list: list of the cost function value
        x_list: list of the state
        u_list: list of the control
        alpha: the weight of the running cost term
        func: the function used to incoporate the RDP inequality into the loss function
        test: whether to print the RDP value
    Return:
        RDP criteria
    """
    Q = torch.diag(q)
    R = r
    F = f
    loss = 0
    for i in range(mpc_T - 1):
        RDP = VN_list[i+1] + alpha * cost_cartpole(x_list[i], u_list[i], Q, R, F, False) - VN_list[i] # Wish RDP <= 0

        if test:
            if log_path is not None:
                with open(log_path, 'a') as f:
                    f.write(f'RDP{i}: {RDP}\n')
        loss += func(RDP) 
        
    return loss

# Training

In [105]:
n_batch = 4
T = 3
mpc_T = 4
epochs = 100
DTYPE = torch.float32
DEVICE = 'cpu'

set_seed(42)
q = nn.Parameter(torch.randn(5, device=DEVICE))
r = torch.tensor([0.001], device=DEVICE)
f = nn.Parameter(torch.randn((5, 5), device=DEVICE))

loss_list = []
optimizer = torch.optim.Adam([q, f], lr=1e-2) # Adam optimizer to update q and f parameters
for epoch in range(epochs):
    states, actions, objs = solve_mpc_cartpole(n_batch, T, mpc_T, q, r, f, DTYPE, DEVICE)
    print(states)
    V_list = cal_V(states, actions, q, r, f, mpc_T, T)
    x_list = [nom_states[0] for nom_states in states]
    u_list = [nom_actions[0] for nom_actions in actions]
    print(objs)
    print(V_list)
    optimizer.zero_grad()
    rdp_loss = RDP_loss(V_list, x_list, u_list, q, r, f, 1, mpc_T, lambda x:max_square(x, 1e-6))
    print(rdp_loss)

100%|██████████| 3/3 [00:00<00:00,  3.31it/s]

[tensor([[[-0.3413, -0.1042, -1.8452,  0.5344, -0.0087],
         [ 0.1542,  0.4147, -0.3822,  0.7863,  0.3913],
         [-0.1722, -0.2964, -0.7337,  0.9639, -0.3553],
         [ 0.1532, -0.2982, -0.4543,  0.8380,  0.0315]],

        [[-0.3465,  0.2879, -1.8450,  0.5348,  0.8811],
         [ 0.1749, -3.6746, -0.3977,  0.7983,  4.7588],
         [-0.1870, -1.9684, -0.7167,  0.9590,  1.0210],
         [ 0.1383, -4.8629, -0.4556,  0.8388,  4.3838]],

        [[-0.3321, -0.9950, -1.8677,  0.4970, -0.3519],
         [-0.0088, -7.3025, -0.6028,  0.9178,  8.6230],
         [-0.2854, -2.6290, -0.7660,  0.9722,  2.0067],
         [-0.1048, -3.7075, -0.6510,  0.9371,  4.0569]],

        [[-0.3818,  0.3975, -1.8589,  0.5122,  1.8259],
         [-0.3739, -2.5632, -1.0227,  0.9997,  6.4742],
         [-0.4169, -1.6792, -0.8645,  0.9908,  2.3878],
         [-0.2902,  0.9004, -0.8470,  0.9882,  2.3337]]],
       grad_fn=<LQRStepFnBackward>), tensor([[[-3.4648e-01,  2.8787e-01, -1.8450e+00,  5.3476e-

IndexError: list index out of range

In [ ]:
dx = cartpole.CartpoleDx()

n_batch, T, mpc_T = 32, 50, 50

# 用于生成均匀分布
def uniform(shape, low, high):
    r = high-low
    return torch.rand(shape)*r+low

torch.manual_seed(0)
# 初始状态
th = uniform(n_batch, -2*np.pi, 2*np.pi)
thdot = uniform(n_batch, -.5, .5)
x = uniform(n_batch, -0.5, 0.5)
xdot = uniform(n_batch, -0.5, 0.5)
xinit = torch.stack((x, xdot, torch.cos(th), torch.sin(th), thdot), dim=1)
print(xinit)

x = xinit
u_init = None

q, p = dx.get_true_obj()
print(q)
print(p)

# 每个batch和每个timestamp的运行损失都是一样的，这里就是在构造这样的Q和p
Q = torch.diag(q).unsqueeze(0).unsqueeze(0).repeat(
    mpc_T, n_batch, 1, 1
)
p = p.unsqueeze(0).repeat(mpc_T, n_batch, 1)

t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

action_history = []
for t in tqdm(range(T)):
    nominal_states, nominal_actions, nominal_objs = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        u_init=u_init,                                  # u_init的作用: warm-start，表示对控制序列的初始猜测 (用上一个时刻的预测结果)，可以加速收敛
        # u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(x, QuadCost(Q, p), dx)
    
    next_action = nominal_actions[0]
    action_history.append(next_action)
    u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
    u_init[-2] = u_init[-3] # 不知道为什么这样写，猜想应该是u[-1] = u[-2]才对

    x = dx(x, next_action) # 往前走一步

    # 以下都是用来可视化的
    n_col = 4
    n_row = n_batch // n_col
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        dx.get_frame(x[i], ax=axs[i])
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
    plt.close(fig)
    
action_history = torch.stack(action_history).detach()[:,:,0]

In [ ]:
# Plot actions
for t in tqdm(range(T)):
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        axs[i].plot(action_history[:,i], color='k')
        axs[i].set_ylim(-15, 15)
        axs[i].axvline(t, color='k', ls='--', linewidth=4)
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'actions_{:03d}.png'.format(t)))
    plt.close(fig)
    
    f1 = os.path.join(t_dir, 'frame_{:03d}.png'.format(t))
    f2 = os.path.join(t_dir, 'actions_{:03d}.png'.format(t))
    f_out = os.path.join(t_dir, '{:03d}.png'.format(t))
    os.system(f'convert {f1} {f2} +append -resize 1200x {f_out}')

In [ ]:
vid_fname = 'cartpole.mp4'

if os.path.exists(vid_fname):
    os.remove(vid_fname)
    
t_dir = 'D:/Docs/code_lib/graduation_test/cartpole_pic'

cmd = 'ffmpeg -r 16 -f image2 -i {}/frame_%03d.png -vcodec libx264 -crf 25 -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -pix_fmt yuv420p {}'.format(
    t_dir, vid_fname
)
print(cmd)
os.system(cmd)
print('Saving video to: {}'.format(vid_fname))

In [ ]:
video = io.open(vid_fname, 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''<video alt="test" controls>
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii')))